In [ ]:
"""
JPMorganChase Inspired Fraud Alert Prioritization
Step 1: Import required libraries
"""
%pip install pandas numpy scikit-learn
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    precision_recall_curve,
    roc_auc_score
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:

# Step 2: Load datasets using folder path

path = r"D:\NextLeap\Assignment Phase2\JP Morgan\02_JPMorgan_Explainable_Fraud_Alert_Prioritization_Project_Kit-20260505T141007Z-3-001\02_JPMorgan_Explainable_Fraud_Alert_Prioritization_Project_Kit\B_Dataset\raw"

transactions = pd.read_csv(path + r"\transactions.csv")
customers   = pd.read_csv(path + r"\customers.csv")
merchants   = pd.read_csv(path + r"\merchants.csv")

print("Transactions:", transactions.shape)
print("Customers:", customers.shape)
print("Merchants:", merchants.shape)

In [ ]:
# =========================================================
# STEP 3: QUICK DATA INSPECTION

#  first inspect the datasets before cleaning or modeling.
# - available columns
# - missing values
# - data types
# - fraud imbalance
# - possible data quality issues

# Preview first 5 rows of transaction data

print("\n================ TRANSACTIONS DATA ================\n")

print(transactions.head())


# Preview customer dataset
print("\n================ CUSTOMERS DATA ================\n")

print(customers.head())

# Preview merchant dataset
print("\n================ MERCHANTS DATA ================\n")

print(merchants.head())

# Check transaction dataset structure
print("\n================ TRANSACTIONS INFO ================\n")

# info() shows:
# - column names
# - data types
# - non-null counts
# - memory usage
print(transactions.info())



In [ ]:
 #---------------------------------------------------------
# Check fraud distribution
# ---------------------------------------------------------
print("\n================ FRAUD DISTRIBUTION ================\n")

# value_counts() tells us how many fraud vs non-fraud cases exist
fraud_counts = transactions["fraud_label"].value_counts()

print(fraud_counts)

# ---------------------------------------------------------
# Calculate fraud percentage
# ---------------------------------------------------------
fraud_percentage = (
    transactions["fraud_label"].mean() * 100
)

print("\nFraud Percentage:")
print(f"{fraud_percentage:.2f}%")

In [39]:
# =========================================================
# STEP 4: DATA CLEANING
# =========================================================

# Goal:
# Clean customer and merchant datasets before merging.
# Fraud models are very sensitive to dirty data.


# =========================================================
# FIX TENURE MONTHS CALCULATION
# =========================================================

# Use latest transaction date instead of today's date

reference_date = pd.to_datetime(
    transactions["event_ts"]
).max()

customers["tenure_months"] = (
    reference_date -
    pd.to_datetime(customers["tenure_months"])
).dt.days // 30

# Check results
print(customers["tenure_months"].describe())


count    5000.0
mean      673.0
std         0.0
min       673.0
25%       673.0
50%       673.0
75%       673.0
max       673.0
Name: tenure_months, dtype: float64


In [ ]:
# ---------------------------------------------------------
# Avoid duplicate column names after merging
# ---------------------------------------------------------

# Both transaction and merchant tables contains
# similar risk column names.

# Rename merchant risk column to keep things clear.

merchants = merchants.rename(
    columns={
        "merchant_risk_score": "merchant_risk_score_current"
    }
)

print("\nMerchant columns renamed successfully.")


In [ ]:
# ---------------------------------------------------------
# Check missing values before merge
# ---------------------------------------------------------

print("\n================ MISSING VALUES ================\n")

print(customers.isnull().sum())

print("\nMerchant Missing Values:\n")

print(merchants.isnull().sum())

In [ ]:
# =========================================================
# STEP 5: MERGE DATASETS
# =========================================================

# Goal:
# Create one master analytical table for fraud modeling.

# combine:
# - transaction data
# - customer information
# - merchant information


# ---------------------------------------------------------
# Merge transactions with customer data
# ---------------------------------------------------------

# how="left" keeps all transactions
# even if some customer records are missing

df = transactions.merge(
    customers,
    on="customer_id",
    how="left"
)

print("\nAfter customer merge:")
print(df.shape)


# ---------------------------------------------------------
# Merge merchant data
# ---------------------------------------------------------

df = df.merge(
    merchants,
    on="merchant_id",
    how="left"
)

print("\nAfter merchant merge:")
print(df.shape)


# ---------------------------------------------------------
# Check missing values after merge
# ---------------------------------------------------------

print("\n================ NULL VALUES AFTER MERGE ================\n")

print(df.isnull().sum())


# ---------------------------------------------------------
# Quick preview of final master table
# ---------------------------------------------------------

print("\n================ MASTER TABLE PREVIEW ================\n")

print(df.head())

In [ ]:
# =========================================================
# STEP 6: HANDLE MISSING VALUES
# =========================================================

# Goal:
# Ensure the fraud pipeline is stable even if some
# customer or merchant records are missing in future data.


# ---------------------------------------------------------
# Check total missing values
# ---------------------------------------------------------

print("\n================ TOTAL NULL VALUES ================\n")

print(df.isnull().sum().sort_values(ascending=False))


# ---------------------------------------------------------
# Fill numeric missing values
# ---------------------------------------------------------

# Median is safer than mean for fraud data
# because fraud datasets often contain outliers.

numeric_cols = [
    "age",
    "tenure_months",
    "synthetic_identity_score",
    "merchant_risk_score"
]

for col in numeric_cols:
    
    if col in df.columns:
        
        df[col] = df[col].fillna(
            df[col].median()
        )


# ---------------------------------------------------------
# Fill categorical missing values
# ---------------------------------------------------------

categorical_cols = [
    "home_country",
    "kyc_risk_band",
    "segment"
]

for col in categorical_cols:
    
    if col in df.columns:
        
        df[col] = df[col].fillna("Unknown")


# ---------------------------------------------------------
# Convert KYC risk into numeric values
# ---------------------------------------------------------

# Machine learning models work better with numeric encoding.

df["kyc_risk_band"] = df["kyc_risk_band"].map({
    "Low": 0,
    "Medium": 1,
    "High": 2,
    "Unknown": 0
})


# ---------------------------------------------------------
# Final null check
# ---------------------------------------------------------

print("\n================ NULL VALUES AFTER CLEANING ================\n")

print(df.isnull().sum().sum())

In [ ]:
# =========================================================
# STEP 6 FIX: HANDLE REMAINING NULLS
# =========================================================

# digital_only is binary:
# 1 = digital customer
# 0 = non-digital customer

# Missing values are safely treated as 0.

df["digital_only"] = df["digital_only"].fillna(0)


# Convert to integer
df["digital_only"] = df["digital_only"].astype(int)


# ---------------------------------------------------------
# Final null check again
# ---------------------------------------------------------

print("\n================ FINAL NULL CHECK ================\n")

print(df.isnull().sum().sum())

In [ ]:
# =========================================================
# STEP 7: FEATURE ENGINEERING
# =========================================================

# Goal:
# Create fraud-related behavioral features that help
# the model detect suspicious activity patterns.


# ---------------------------------------------------------
# Convert timestamp column into datetime format
# ---------------------------------------------------------

# Fraud analysis depends heavily on time patterns.

df["event_ts"] = pd.to_datetime(df["event_ts"])


# ---------------------------------------------------------
# Velocity Ratio
# ---------------------------------------------------------

# Measures sudden transaction bursts.

# Example:
# If a customer normally makes few transactions
# but suddenly makes many within 1 hour,
# fraud risk increases.

# replace(0,1) prevents division by zero.

df["velocity_ratio"] = (
    df["velocity_1h"] /
    df["velocity_24h"].replace(0, 1)
)


In [ ]:
# ---------------------------------------------------------
# Geographic Mismatch Flag
# ---------------------------------------------------------

# Checks if transaction country differs from
# customer's registered home country.

# International mismatch can indicate:
# - account takeover
# - stolen cards
# - proxy/VPN abuse

df["geo_mismatch_flag"] = (
    df["txn_country"] != df["home_country"]
).astype(int)


In [ ]:
# ---------------------------------------------------------
# Log Transaction Amount
# ---------------------------------------------------------

# Fraud datasets usually contain highly skewed
# transaction amounts.

# log1p reduces extreme outlier impact
# while preserving order.

df["amount_log"] = np.log1p(
    df["transaction_amount_usd"]
)



In [ ]:
# ---------------------------------------------------------
# Composite Risk Score
# ---------------------------------------------------------

# Combines multiple risk signals into one feature.

# This simulates how real fraud systems often
# aggregate internal risk indicators.

df["composite_risk"] = (
    df["device_risk_score"] +
    df["merchant_risk_score"] +
    df["kyc_risk_band"] +
    df["synthetic_identity_score"]
)

In [ ]:

# ---------------------------------------------------------
# Encode Channel Variable
# ---------------------------------------------------------

# Machine learning models require numeric inputs.

# Convert transaction channel into dummy variables.

# drop_first=True avoids redundant columns.

df = pd.get_dummies(
    df,
    columns=["channel"],
    drop_first=True
)


In [ ]:

# ---------------------------------------------------------
# Preview engineered features
# ---------------------------------------------------------

print("\n================ FEATURE PREVIEW ================\n")

feature_preview = [
    "velocity_ratio",
    "geo_mismatch_flag",
    "amount_log",
    "composite_risk"
]

print(df[feature_preview].head())

In [40]:
# =========================================================
# STEP 8: TRAIN-TEST SPLIT
# =========================================================

# Goal:
# Split historical transactions into:
# - training data
# - future test data

# IMPORTANT:
# ths Fraud systems do not use random train_test_split().
# That causes data leakage.

# ---------------------------------------------------------
# Sort data by transaction timestamp
# ---------------------------------------------------------
# Ensures proper historical ordering.

df = df.sort_values(
    "event_ts"
).reset_index(drop=True)


# ---------------------------------------------------------
# Create time-based split
# ---------------------------------------------------------

# Train on older transactions
# Test on newer transactions

train = df[
    df["event_ts"] < "2025-03-01"
]

test = df[
    df["event_ts"] >= "2025-03-01"
]

# ---------------------------------------------------------
# Check dataset sizes
# ---------------------------------------------------------

print("\n================ DATASET SPLIT ================\n")

print("Training rows :", len(train))
print("Testing rows  :", len(test))



================ DATASET SPLIT ================

Training rows : 2420
Testing rows  : 2580


In [41]:
# ---------------------------------------------------------
# Select model features
# ---------------------------------------------------------

# These are the strongest fraud indicators identified
# during feature engineering and exploratory analysis.

feature_cols = [

    # Transaction behavior
    "transaction_amount_usd",
    "amount_log",

    # Device and behavioral risk
    "device_risk_score",
    "new_device_flag",
    "velocity_1h",
    "velocity_ratio",

    # Geographic behavior
    "geo_distance_km",
    "geo_mismatch_flag",

    # Composite risk indicators
    "composite_risk",
    "synthetic_identity_score",
    "kyc_risk_band",

    
    # Time behavior
    "is_night_flag",

    # Transaction channels
    "channel_P2P",
    "channel_Wire"
]


In [42]:
# ---------------------------------------------------------
# Build X and y datasets
# ---------------------------------------------------------

# X = model inputs
# y = fraud target variable

X_train = train[feature_cols]
y_train = train["fraud_label"]

X_test = test[feature_cols]
y_test = test["fraud_label"]


# ---------------------------------------------------------
# Verify shapes
# ---------------------------------------------------------

print("\n================ FEATURE MATRIX ================\n")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nFraud cases in training:")
print(y_train.value_counts())

print("\nFraud cases in testing:")
print(y_test.value_counts())


================ FEATURE MATRIX ================

X_train: (2420, 14)
X_test : (2580, 14)

Fraud cases in training:
fraud_label
0    2317
1     103
Name: count, dtype: int64

Fraud cases in testing:
fraud_label
0    2482
1      98
Name: count, dtype: int64


In [43]:
# =========================================================
# STEP 9: TRAIN FRAUD MODEL
# =========================================================

# Goal:
# Train a fraud detection model using historical data.


# ---------------------------------------------------------
# Create Random Forest model
# ---------------------------------------------------------

# Why Random Forest?
# - Handles fraud patterns well
# - Robust to noisy data
# - Easy to explain
# - Works well on structured datasets

# class_weight="balanced"
# helps the model focus more on rare fraud cases.

model = RandomForestClassifier(

    # Number of trees
    n_estimators=200,

    # Prevents overly complex trees
    max_depth=6,

    # Helps handle fraud imbalance
    class_weight="balanced",

    # Makes results reproducible
    random_state=42,

    # Uses all CPU cores
    n_jobs=-1
)


# ---------------------------------------------------------
# Train the model
# ---------------------------------------------------------

# The model learns fraud patterns from historical transactions.

model.fit(X_train, y_train)

print("\nModel training completed successfully.")


# ---------------------------------------------------------
# Generate fraud probabilities
# ---------------------------------------------------------

# predict_proba() gives fraud probability scores.

# [:,1] means probability of FRAUD class.

probs = model.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------
# Preview probabilities
# ---------------------------------------------------------

print("\n================ FRAUD PROBABILITIES ================\n")

print(probs[:10])


Model training completed successfully.

================ FRAUD PROBABILITIES ================

[0.2574726  0.30094617 0.39427499 0.36746266 0.34057397 0.22367774
 0.53908464 0.33470123 0.43881699 0.40600907]


In [44]:
# =========================================================
# STEP 10: FIND BEST FRAUD THRESHOLD
# =========================================================

# Goal:
# Find the smartest probability cutoff for fraud alerts.


# ---------------------------------------------------------
# Generate precision-recall curve
# ---------------------------------------------------------

# Precision:
# Of all predicted frauds,
# how many were actually fraud?

# Recall:
# Of all actual frauds,
# how many did we catch?

precision_curve, recall_curve, thresholds = (
    precision_recall_curve(y_test, probs)
)


# ---------------------------------------------------------
# Calculate F1 scores
# ---------------------------------------------------------

# F1 score balances:
# - precision
# - recall

# This is very important in fraud analytics.

f1_scores = (
    2 * precision_curve * recall_curve
) / (
    precision_curve + recall_curve + 1e-9
)


# ---------------------------------------------------------
# Find best threshold
# ---------------------------------------------------------

best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]

print("\n================ BEST THRESHOLD ================\n")

print(f"Best Threshold : {best_threshold:.3f}")

print(f"Best F1 Score  : {f1_scores[best_index]:.3f}")


# ---------------------------------------------------------
# Convert probabilities into fraud predictions
# ---------------------------------------------------------

# If probability >= threshold:
# classify as fraud

preds = (
    probs >= best_threshold
).astype(int)


# ---------------------------------------------------------
# Preview predictions
# ---------------------------------------------------------

print("\n================ SAMPLE PREDICTIONS ================\n")

preview = pd.DataFrame({

    "Fraud_Probability": probs[:10],
    "Predicted_Fraud": preds[:10]

})

print(preview)


================ BEST THRESHOLD ================

Best Threshold : 0.422
Best F1 Score  : 0.132

================ SAMPLE PREDICTIONS ================

   Fraud_Probability  Predicted_Fraud
0           0.257473                0
1           0.300946                0
2           0.394275                0
3           0.367463                0
4           0.340574                0
5           0.223678                0
6           0.539085                1
7           0.334701                0
8           0.438817                1
9           0.406009                0


In [45]:
# =========================================================
# STEP 11: MODEL EVALUATION
# =========================================================

# Goal:
# Evaluate how well the fraud model performs.


# ---------------------------------------------------------
# Classification Report
# ---------------------------------------------------------

# Shows:
# - precision
# - recall
# - F1 score
# for both fraud and non-fraud classes.

print("\n================ CLASSIFICATION REPORT ================\n")

print(
    classification_report(
        y_test,
        preds,
        target_names=[
            "Not Fraud",
            "Fraud"
        ]
    )
)


# ---------------------------------------------------------
# ROC-AUC Score
# ---------------------------------------------------------

# ROC-AUC measures the model's ability
# to separate fraud from non-fraud.

# Higher is better:
# 0.50 = random guessing
# 1.00 = perfect model

auc = roc_auc_score(
    y_test,
    probs
)

print("\n================ ROC-AUC SCORE ================\n")

print(f"ROC-AUC : {auc:.3f}")


# ---------------------------------------------------------
# Count predicted fraud alerts
# ---------------------------------------------------------

predicted_fraud = preds.sum()

actual_fraud = y_test.sum()

print("\n================ ALERT SUMMARY ================\n")

print(f"Actual Fraud Cases     : {actual_fraud}")

print(f"Predicted Fraud Alerts : {predicted_fraud}")


# ---------------------------------------------------------
# Fraud detection rate
# ---------------------------------------------------------

# Measures how much fraud we successfully caught.

fraud_caught = preds[y_test == 1].sum()

fraud_recall = (
    fraud_caught / actual_fraud
) * 100

print(f"Fraud Detection Rate   : {fraud_recall:.1f}%")


================ CLASSIFICATION REPORT ================

              precision    recall  f1-score   support

   Not Fraud       0.97      0.93      0.95      2482
       Fraud       0.10      0.19      0.13        98

    accuracy                           0.90      2580
   macro avg       0.53      0.56      0.54      2580
weighted avg       0.93      0.90      0.92      2580


================ ROC-AUC SCORE ================

ROC-AUC : 0.609

================ ALERT SUMMARY ================

Actual Fraud Cases     : 98
Predicted Fraud Alerts : 189
Fraud Detection Rate   : 19.4%


In [46]:
# =========================================================
# STEP 12: ADVANCED FRAUD FEATURES
# =========================================================

# Goal:
# Add stronger behavioral fraud signals
# to improve fraud recall and fraud ranking.


# ---------------------------------------------------------
# Customer Average Transaction Amount
# ---------------------------------------------------------

# Calculate each customer's normal spending level.

customer_avg_amount = (
    df.groupby("customer_id")["transaction_amount_usd"]
    .transform("mean")
)

df["customer_avg_amount"] = customer_avg_amount


# ---------------------------------------------------------
# Transaction vs Customer Average
# ---------------------------------------------------------

# Measures how unusual the current transaction is.

# Example:
# Customer normally spends $50
# Current transaction = $500

# High anomaly ratio = suspicious.

df["txn_vs_customer_avg"] = (
    df["transaction_amount_usd"] /
    (df["customer_avg_amount"] + 1)
)


# ---------------------------------------------------------
# Customer Transaction Count
# ---------------------------------------------------------

# Number of transactions made by each customer.

# Fraudsters often create abnormal activity bursts.

customer_txn_count = (
    df.groupby("customer_id")["transaction_id"]
    .transform("count")
)

df["customer_txn_count"] = customer_txn_count


# ---------------------------------------------------------
# Merchant Fraud Rate
# ---------------------------------------------------------

# Historical fraud rate for each merchant.

# Some merchants naturally have higher fraud exposure.

merchant_fraud_rate = (
    df.groupby("merchant_id")["fraud_label"]
    .transform("mean")
)

df["merchant_fraud_rate"] = merchant_fraud_rate


# ---------------------------------------------------------
# High-Risk Transaction Hour
# ---------------------------------------------------------

# Late-night transactions are often riskier.

# Create a stronger night-risk signal.

df["high_risk_hour_flag"] = (
    (df["txn_hour"] >= 23) |
    (df["txn_hour"] <= 4)
).astype(int)


# ---------------------------------------------------------
# Preview new fraud features
# ---------------------------------------------------------

print("\n================ NEW FRAUD FEATURES ================\n")

new_features = [

    "customer_avg_amount",
    "txn_vs_customer_avg",
    "customer_txn_count",
    "merchant_fraud_rate",
    "high_risk_hour_flag"

]

print(df[new_features].head())


================ NEW FRAUD FEATURES ================

   customer_avg_amount  txn_vs_customer_avg  customer_txn_count  \
0                15.53             0.939504                   1   
1                26.45             0.963570                   1   
2               152.53             0.993487                   1   
3                71.71             0.986247                   1   
4                38.25             0.974522                   1   

   merchant_fraud_rate  high_risk_hour_flag  
0                  0.0                    0  
1                  0.0                    0  
2                  0.0                    0  
3                  0.0                    1  
4                  0.0                    0  


In [47]:
# =========================================================
# STEP 13 FIX: RECREATE TRAIN AND TEST AFTER NEW FEATURES
# =========================================================

# We added new fraud features to df.
# So we must recreate train and test from the updated df.

df = df.sort_values("event_ts").reset_index(drop=True)

train = df[df["event_ts"] < "2025-03-01"]
test  = df[df["event_ts"] >= "2025-03-01"]

print("Updated train shape:", train.shape)
print("Updated test shape :", test.shape)

# Check if new features now exist inside train
new_features = [
    "customer_avg_amount",
    "txn_vs_customer_avg",
    "customer_txn_count",
    "merchant_fraud_rate",
    "high_risk_hour_flag"
]

print("\nNew feature check:")
print(train[new_features].head())

Updated train shape: (2420, 39)
Updated test shape : (2580, 39)

New feature check:
   customer_avg_amount  txn_vs_customer_avg  customer_txn_count  \
0                15.53             0.939504                   1   
1                26.45             0.963570                   1   
2               152.53             0.993487                   1   
3                71.71             0.986247                   1   
4                38.25             0.974522                   1   

   merchant_fraud_rate  high_risk_hour_flag  
0                  0.0                    0  
1                  0.0                    0  
2                  0.0                    0  
3                  0.0                    1  
4                  0.0                    0  


In [48]:
# =========================================================
# STEP 14: RETRAIN MODEL WITH ADVANCED FEATURES
# =========================================================

feature_cols = [
    "transaction_amount_usd",
    "amount_log",
    "customer_avg_amount",
    "txn_vs_customer_avg",

    "device_risk_score",
    "new_device_flag",
    "velocity_1h",
    "velocity_ratio",

    "geo_distance_km",
    "geo_mismatch_flag",

    "composite_risk",
    "merchant_fraud_rate",
    "synthetic_identity_score",
    "kyc_risk_band",

    "tenure_months",
    "customer_txn_count",

    "is_night_flag",
    "high_risk_hour_flag",

    "channel_P2P",
    "channel_Wire"
]

X_train = train[feature_cols]
y_train = train["fraud_label"]

X_test = test[feature_cols]
y_test = test["fraud_label"]

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]

precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, probs)

f1_scores = (
    2 * precision_curve * recall_curve
) / (
    precision_curve + recall_curve + 1e-9
)

best_index = np.argmax(f1_scores)
best_threshold = thresholds[best_index]

preds = (probs >= best_threshold).astype(int)

print("\nImproved fraud model trained successfully.")
print(f"Best Threshold : {best_threshold:.3f}")
print(f"Best F1 Score  : {f1_scores[best_index]:.3f}")


Improved fraud model trained successfully.
Best Threshold : 0.682
Best F1 Score  : 1.000


In [49]:
# =========================================================
# STEP 14: FIX DATA LEAKAGE
# =========================================================

# Goal:
# Remove target leakage from merchant fraud rate.


# ---------------------------------------------------------
# Recreate clean train and test split
# ---------------------------------------------------------

df = df.sort_values("event_ts").reset_index(drop=True)

train = df[df["event_ts"] < "2025-03-01"].copy()
test  = df[df["event_ts"] >= "2025-03-01"].copy()


# ---------------------------------------------------------
# Calculate merchant fraud rate ONLY from training data
# ---------------------------------------------------------

# This prevents future fraud information leakage.

merchant_fraud_lookup = (
    train.groupby("merchant_id")["fraud_label"]
    .mean()
)


# ---------------------------------------------------------
# Map fraud rates separately
# ---------------------------------------------------------

train["merchant_fraud_rate"] = (
    train["merchant_id"]
    .map(merchant_fraud_lookup)
)

test["merchant_fraud_rate"] = (
    test["merchant_id"]
    .map(merchant_fraud_lookup)
)


# ---------------------------------------------------------
# Fill unseen merchants in test set
# ---------------------------------------------------------

# New merchants may not exist in training history.

global_fraud_rate = train["fraud_label"].mean()

test["merchant_fraud_rate"] = (
    test["merchant_fraud_rate"]
    .fillna(global_fraud_rate)
)


# ---------------------------------------------------------
# Final null check
# ---------------------------------------------------------

print("\nTrain nulls:")
print(train["merchant_fraud_rate"].isnull().sum())

print("\nTest nulls:")
print(test["merchant_fraud_rate"].isnull().sum())


Train nulls:
0

Test nulls:
0


In [51]:
# =========================================================
# STEP 15: RETRAIN LEAKAGE-FREE FRAUD MODEL
# =========================================================

# Goal:
# Train a realistic fraud model without target leakage.


# ---------------------------------------------------------
# Feature list
# ---------------------------------------------------------

feature_cols = [

    # Transaction amount behavior
    "transaction_amount_usd",
    "amount_log",
    "customer_avg_amount",
    "txn_vs_customer_avg",

    # Device and velocity behavior
    "device_risk_score",
    "new_device_flag",
    "velocity_1h",
    "velocity_ratio",

    # Geographic behavior
    "geo_distance_km",
    "geo_mismatch_flag",

    # Risk indicators
    "composite_risk",
    "merchant_fraud_rate",
    "synthetic_identity_score",
    "kyc_risk_band",

    # Customer profile
   
    "customer_txn_count",

    # Time behavior
    "is_night_flag",
    "high_risk_hour_flag",

    # Channel indicators
    "channel_P2P",
    "channel_Wire"
]


# ---------------------------------------------------------
# Build datasets
# ---------------------------------------------------------

X_train = train[feature_cols]
y_train = train["fraud_label"]

X_test = test[feature_cols]
y_test = test["fraud_label"]


# ---------------------------------------------------------
# Train Random Forest model
# ---------------------------------------------------------

model = RandomForestClassifier(

    n_estimators=300,

    max_depth=8,

    min_samples_leaf=5,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1
)

model.fit(X_train, y_train)

print("\nLeakage-free fraud model trained successfully.")


# ---------------------------------------------------------
# Generate fraud probabilities
# ---------------------------------------------------------

probs = model.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------
# Threshold optimization
# ---------------------------------------------------------

precision_curve, recall_curve, thresholds = (
    precision_recall_curve(y_test, probs)
)

f1_scores = (
    2 * precision_curve * recall_curve
) / (
    precision_curve + recall_curve + 1e-9
)

best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]

preds = (
    probs >= best_threshold
).astype(int)


# ---------------------------------------------------------
# Print evaluation metrics
# ---------------------------------------------------------

print("\n================ FINAL MODEL RESULTS ================\n")

print(f"Best Threshold : {best_threshold:.3f}")

print(f"Best F1 Score  : {f1_scores[best_index]:.3f}")


# ---------------------------------------------------------
# ROC-AUC
# ---------------------------------------------------------

auc = roc_auc_score(y_test, probs)

print(f"ROC-AUC Score  : {auc:.3f}")


# ---------------------------------------------------------
# Classification report
# ---------------------------------------------------------

print("\n================ CLASSIFICATION REPORT ================\n")

print(
    classification_report(
        y_test,
        preds,
        target_names=[
            "Not Fraud",
            "Fraud"
        ]
    )
)


Leakage-free fraud model trained successfully.

================ FINAL MODEL RESULTS ================

Best Threshold : 0.086
Best F1 Score  : 0.162
ROC-AUC Score  : 0.637

================ CLASSIFICATION REPORT ================

              precision    recall  f1-score   support

   Not Fraud       0.97      0.93      0.95      2482
       Fraud       0.12      0.24      0.16        98

    accuracy                           0.90      2580
   macro avg       0.55      0.59      0.56      2580
weighted avg       0.94      0.90      0.92      2580



In [52]:
# =========================================================
# STEP 17: REALISTIC FRAUD QUEUE THRESHOLDS
# =========================================================

# Goal:
# Build operational fraud queues using realistic
# probability thresholds based on model behavior.


# ---------------------------------------------------------
# Create improved priority tiers
# ---------------------------------------------------------

# High Priority:
# immediate investigation

# Medium Priority:
# analyst review

# Low Priority:
# monitor only

# =========================================================
# RECREATE RESULT DATASET
# =========================================================

# Create a fresh result table from test data

result = test.copy()

# Add fraud probabilities from the model
result["fraud_probability"] = probs

print("Result dataset recreated successfully.")

# Quick preview
print(result[[
    "transaction_id",
    "fraud_probability",
    "fraud_label"
]].head())


result["priority_tier"] = np.select(

    [
        result["fraud_probability"] >= 0.15,
        result["fraud_probability"] >= 0.07
    ],

    [
        "High Priority",
        "Medium Priority"
    ],

    default="Low Priority"
)



Result dataset recreated successfully.
     transaction_id  fraud_probability  fraud_label
2420       T9005394           0.049776            0
2421       T9004129           0.022997            0
2422       T9038245           0.070603            0
2423       T9009035           0.070208            0
2424       T9009485           0.090405            0


In [53]:

# ---------------------------------------------------------
# Queue summary
# ---------------------------------------------------------

queue_summary = (

    result
    .groupby("priority_tier")
    .agg(

        total_alerts=(
            "transaction_id",
            "count"
        ),

        confirmed_fraud=(
            "fraud_label",
            "sum"
        )

    )
)



In [54]:

# ---------------------------------------------------------
# Calculate hit rate
# ---------------------------------------------------------

queue_summary["hit_rate_percent"] = (

    queue_summary["confirmed_fraud"] /
    queue_summary["total_alerts"]

) * 100


queue_summary["hit_rate_percent"] = (
    queue_summary["hit_rate_percent"]
    .round(2)
)


In [55]:
# ---------------------------------------------------------
# Display final queue summary
# ---------------------------------------------------------

print("\n================ FINAL FRAUD QUEUE SUMMARY ================\n")

print(queue_summary)


# ---------------------------------------------------------
# Alert distribution
# ---------------------------------------------------------

print("\n================ ALERT DISTRIBUTION ================\n")

print(
    result["priority_tier"]
    .value_counts()
)


================ FINAL FRAUD QUEUE SUMMARY ================

                 total_alerts  confirmed_fraud  hit_rate_percent
priority_tier                                                   
High Priority              11                1              9.09
Low Priority             2140               61              2.85
Medium Priority           429               36              8.39

================ ALERT DISTRIBUTION ================

priority_tier
Low Priority       2140
Medium Priority     429
High Priority        11
Name: count, dtype: int64


In [56]:
# =========================================================
# STEP 1: EXPORT FINAL DATASET FOR POWER BI
# =========================================================

# We use the final result table because it already contains:
# - original transaction data
# - fraud probability
# - priority tier
# - fraud label
# - customer and merchant details

powerbi_export = result.copy()

# Export the final dataset as CSV
powerbi_export.to_csv(
    "fraud_dashboard_dataset.csv",
    index=False
)

print("Power BI dataset exported successfully.")
import os

print(os.getcwd())

Power BI dataset exported successfully.
c:\Users\mon12\.jupyter
